In [ ]:
pip install psycopg2-binary

In [ ]:
import json
import psycopg2
from psycopg2.extras import execute_values

DATA_PATH = "data_export/teachers.json"

CONN = dict(
    host="localhost",
    port=5432,
    dbname="eoperasi",
    user="eoperasi",
    password="devpassword123",
)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    teachers = json.load(f)

print(f"Loaded {len(teachers)} teacher records")

In [ ]:
conn = psycopg2.connect(**CONN)
conn.autocommit = False  # we control commits manually, so a bad batch doesn't half-write
cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone())

In [ ]:
cur.execute('''
    TRUNCATE TABLE "TeacherSubjects", "Teachers", "Subjects", "Schools" RESTART IDENTITY CASCADE;
''')
conn.commit()
print("Tables cleared")

In [ ]:
school_rows = {}
for t in teachers:
    key = t["school"]
    if key not in school_rows:
        school_rows[key] = (t["school"], t["schoolType"], t.get("district"))

rows = list(school_rows.values())
print(f"Inserting {len(rows)} distinct schools...")

execute_values(
    cur,
    '''INSERT INTO "Schools" ("Name", "Type", "District") VALUES %s''',
    rows,
)
conn.commit()

cur.execute('''SELECT "Id", "Name" FROM "Schools"''')
school_id_by_name = {name: id_ for id_, name in cur.fetchall()}
print(f"{len(school_id_by_name)} schools now in DB")

In [ ]:
subject_names = set()
for t in teachers:
    for s in t["subjects"]:
        subject_names.add(s["name"])

rows = [(name,) for name in sorted(subject_names)]
print(f"Inserting {len(rows)} distinct subjects...")

execute_values(
    cur,
    '''INSERT INTO "Subjects" ("Name") VALUES %s''',
    rows,
)
conn.commit()

cur.execute('''SELECT "Id", "Name" FROM "Subjects"''')
subject_id_by_name = {name: id_ for id_, name in cur.fetchall()}
print(f"{len(subject_id_by_name)} subjects now in DB")

In [ ]:
INSERT_TEACHER_SQL = '''
INSERT INTO "Teachers" (
    "IcNumber", "Name", "Gender", "DateOfBirth", "MaritalStatus", "Race", "Religion",
    "Nationality", "Phone", "Email", "Address", "SchoolId", "Position", "GredHakiki",
    "GredPenyandang", "EmploymentStatus", "ServiceStatus", "FirstAppointmentDate",
    "Qualification", "Opsyen1", "Opsyen2", "Ikhtisas", "StatusInduksi", "SahPerkhidmatan", "Salary",
    "RetirementDate", "TugasKhas", "GuruCemerlang", "GuruCemerlangBidang", "Vaccinated",
    "VaccineType", "VaccineDose1", "VaccineDose2", "KodJawatanDiisi"
) VALUES (
    %(icNumber)s, %(name)s, %(gender)s, %(dateOfBirth)s, %(maritalStatus)s, %(race)s,
    %(religion)s, %(nationality)s, %(phone)s, %(email)s, %(address)s, %(schoolId)s,
    %(position)s, %(gredHakiki)s, %(gredPenyandang)s, %(employmentStatus)s,
    %(serviceStatus)s, %(firstAppointmentDate)s, %(qualification)s, %(opsyen1)s, %(opsyen2)s, %(ikhtisas)s,
    %(statusInduksi)s, %(sahPerkhidmatan)s, %(salary)s, %(retirementDate)s, %(tugasKhas)s,
    %(guruCemerlang)s, %(guruCemerlangBidang)s, %(vaccinated)s, %(vaccineType)s,
    %(vaccineDose1)s, %(vaccineDose2)s, %(kodJawatanDiisi)s
)
ON CONFLICT ("IcNumber") DO NOTHING
RETURNING "Id";
'''

teacher_db_id_by_ic = {}
skipped = 0

for t in teachers:
    params = dict(t)
    params["schoolId"] = school_id_by_name[t["school"]]
    cur.execute(INSERT_TEACHER_SQL, params)
    result = cur.fetchone()
    if result:
        teacher_db_id_by_ic[t["icNumber"]] = result[0]
    else:
        skipped += 1  # IC already existed (ON CONFLICT DO NOTHING) - re-run case

conn.commit()
print(f"Inserted {len(teacher_db_id_by_ic)} teachers ({skipped} skipped as already present)")

In [ ]:
INSERT_TS_SQL = '''
INSERT INTO "TeacherSubjects" ("TeacherId", "SubjectId", "Hours")
VALUES %s
ON CONFLICT ("TeacherId", "SubjectId") DO NOTHING;
'''

ts_rows = []
for t in teachers:
    teacher_id = teacher_db_id_by_ic.get(t["icNumber"])
    if teacher_id is None:
        continue  # was skipped in step 5 (already existed)
    for s in t["subjects"]:
        subject_id = subject_id_by_name[s["name"]]
        ts_rows.append((teacher_id, subject_id, s["hours"]))

print(f"Inserting {len(ts_rows)} teacher-subject links...")
execute_values(cur, INSERT_TS_SQL, ts_rows)
conn.commit()
print("Done")

In [ ]:
for table in ["Schools", "Subjects", "Teachers", "TeacherSubjects"]:
    cur.execute(f'''SELECT COUNT(*) FROM "{table}"''')
    print(table, "->", cur.fetchone()[0])

cur.close()
conn.close()
print("Connection closed")